<a href="https://colab.research.google.com/github/abhishek24-06/Deepfake-detection-system-for-Images-and-Videos/blob/main/ResNet50%2BLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import os
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_path = "/content/drive/MyDrive/Face_frames"

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for label, category in enumerate(["real","fake"]):
            category_path = os.path.join(root_dir, category)

            for video in os.listdir(category_path):
                video_path = os.path.join(category_path, video)
                self.samples.append((video_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]

        frames = sorted(os.listdir(video_path))[:15]

        images = []

        for frame in frames:
            img_path = os.path.join(video_path, frame)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            if self.transform:
                img = self.transform(img)

            images.append(img)

        images = torch.stack(images)

        return images, torch.tensor(label, dtype=torch.float32)

In [ ]:
train_dataset = DeepfakeDataset(
    "/content/drive/MyDrive/Face_frames/train",
    transform
)

val_dataset = DeepfakeDataset(
    "/content/drive/MyDrive/Face_frames/val",
    transform
)

test_dataset = DeepfakeDataset(
    "/content/drive/MyDrive/Face_frames/test",
    transform
)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
sample, label = train_dataset[0]

print(sample.shape)
print(label)

torch.Size([15, 3, 224, 224])
tensor(0.)


In [ ]:
class DeepfakeModel(nn.Module):

    def __init__(self):
        super().__init__()

        resnet = models.resnet50(pretrained=True)

        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        for param in self.cnn.parameters():
            param.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(128,1)

    def forward(self,x):

        batch, frames, C, H, W = x.size()

        x = x.view(batch*frames, C, H, W)

        features = self.cnn(x)

        features = features.view(batch,frames,2048)

        lstm_out,_ = self.lstm(features)

        output = self.fc(lstm_out[:,-1,:])

        return output

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeepfakeModel().to(device)

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0003
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    # Unfreeze CNN after epoch 5
    if epoch == 5:
        print("Unfreezing last CNN layers...")

        for param in model.cnn[-3:].parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.00001
        )

    model.train()

    running_loss = 0

    for videos, labels in tqdm(train_loader):

        videos = videos.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", running_loss/len(train_loader))

100%|██████████| 400/400 [1:01:21<00:00,  9.20s/it]


Epoch: 1 Loss: 0.6912444102764129


100%|██████████| 400/400 [02:41<00:00,  2.48it/s]


Epoch: 2 Loss: 0.6704403334856033


100%|██████████| 400/400 [02:41<00:00,  2.48it/s]


...

Epoch: 18 Loss: 0.05855788177810609


100%|██████████| 400/400 [02:58<00:00,  2.24it/s]


Epoch: 19 Loss: 0.04069840627955273


100%|██████████| 400/400 [02:57<00:00,  2.26it/s]

Epoch: 20 Loss: 0.03713331325678155


In [ ]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/deepfake_video_model.pth")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        outputs = torch.sigmoid(model(videos))

        preds = (outputs > 0.5).float()

        correct += (preds.squeeze() == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 0.78


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(device)

        outputs = torch.sigmoid(model(videos))
        preds = (outputs > 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

         0.0       0.78      0.78      0.78       100
         1.0       0.78      0.78      0.78       100

    accuracy                           0.78       200
   macro avg       0.78      0.78      0.78       200
weighted avg       0.78      0.78      0.78       200

[[78 22]
 [22 78]]
